In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\AI-book-recommender'

In [5]:
import box
print(box.__version__)

6.0.2


In [6]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DatainjectionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from Book_recommender.constant import *
from Book_recommender.utils.common import read_yaml, create_directories

In [8]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_injection_config(self) -> DatainjectionConfig:
        config = self.config.data_injection

        create_directories([config.root_dir])

        data_injection_config = DatainjectionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_injection_config

In [9]:
import os
import urllib.request as request
import zipfile
from Book_recommender.logging.log import logger
from Book_recommender.utils.common import get_size

In [10]:
# 5 Conponents

class DataInjection:
    def __init__(self, config: DatainjectionConfig):
        self.config = config


    def download_files(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded with following info \n{headers}")
        else:
            logger.info(f"file already exist of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zipfile(self):
        """
        zip_file_path: str
        Extracts the zip file in dir
        None returns
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [15]:
# 6 Pipeline

try:
    config = configurationManager()
    data_injection_config = config.get_data_injection_config()
    data_injection = DataInjection(config = data_injection_config)
    data_injection.download_files()
    data_injection.extract_zipfile()

except Exception as e:
    raise e

[2026-05-27 18:57:50,431: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-05-27 18:57:50,445: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-05-27 18:57:50,447: INFO: common: created directory at artifacts]
[2026-05-27 18:57:50,449: INFO: common: created directory at artifacts/data_injection]
[2026-05-27 18:57:52,355: INFO: 4053066924: artifacts/data_injection/data.zip Downloaded with following info 
Content-Type: application/zip
X-GUploader-UploadID: AAVLpEhDVYSCaUXpT-9BC5cROlGPmEQj2RNhTlqWvYx_ZzpfrINVJmTf-bNYNnkmPqqCg-oO
Expires: Wed, 27 May 2026 17:57:51 GMT
Date: Wed, 27 May 2026 17:57:51 GMT
Cache-Control: private, max-age=0
Last-Modified: Mon, 09 Mar 2020 09:18:34 GMT
ETag: "0879613b15ff78ebe6dd1908e4cf60d8"
x-goog-generation: 1583745514525068
x-goog-metageneration: 1
x-goog-stored-content-encoding: identity
x-goog-stored-content-length: 637338
x-goog-hash: crc32c=RgtyVg==
x-goog-hash: md5=CHlhOxX/eOvm3RkI5M9g2A==
x-goog-storage-c